In [1]:
import os
import numpy as np
from scipy.stats import ks_2samp, ttest_ind
from collections import Counter, defaultdict
import pandas as pd
from statsmodels.tsa.seasonal import STL
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False 

In [2]:
filepath = '../data/사내협력사 현황(철수사&거래 협력사)_ Data_송부_수정.xlsx'
df_raw_data_dict = pd.read_excel(filepath, sheet_name=None)
sheet_name_list = list(df_raw_data_dict.keys())

In [36]:
class Company:
    def __init__(self, company_id, catecory, subcategory, subsubcategory, start_date, end_date, date_range, label, condition):
        self.company_id = company_id
        self.catecory = catecory
        self.subcategory = subcategory
        self.subsubcategory = subsubcategory
        self.start_date = start_date
        self.end_date = end_date
        self.date_range = date_range
        self.label = label
        self.condition = condition
        self.data_dict = dict()

In [37]:
data_start_date = '2016-02-01'
data_end_date = '2024-10-01'
data_duration = 12
label_date = '2024-10-01' # 해당 월 기준으로 계약 종결 여부 판단, 직전월까지의 데이터 사용
data_masking_duration = 2 # masking 기간 

In [38]:
company_dict = dict()
df_temp = df_raw_data_dict[sheet_name_list[0]]
column = list(df_temp.columns)[:8]
if pd.isna(pd.to_datetime(df_temp.columns[-1], errors='coerce')):
    df_temp = df_temp.iloc[:, :-1]
new_cols = (list(df_temp.columns[:9]) + list(pd.to_datetime(df_temp.columns[9:]).map(lambda dt: dt.replace(day=1))))
date_cols = new_cols[9:]

for i, row in df_temp.iterrows():
    r_start_date = pd.to_datetime(row[column[4]]).replace(day=1)
    r_end_date = pd.to_datetime(label_date) - pd.DateOffset(months=1) if row[column[5]] == '-' else pd.to_datetime(row[column[5]]).replace(day=1)
    r_label = True if pd.to_datetime(label_date) >= r_end_date + pd.DateOffset(months=1 - data_masking_duration) else False
    r_label = False if row[column[5]] == '-' else r_label
    if r_label:
        date_range = list(pd.date_range(start=r_end_date - pd.DateOffset(months=data_duration + data_masking_duration - 1), end=r_end_date - pd.DateOffset(months=data_masking_duration), freq='MS'))
    else:
        date_range = list(pd.date_range(start=pd.to_datetime(label_date) - pd.DateOffset(months=data_duration), end=pd.to_datetime(label_date) - pd.DateOffset(months=1), freq='MS'))
    r_condition = all(element in iter(date_cols) for element in date_range) and r_start_date <= date_range[0] and r_end_date >= date_range[-1]
    company_dict[row[column[0]]] = Company(row[column[0]], row[column[1]], row[column[2]], row[column[3]], r_start_date, r_end_date, date_range, r_label, r_condition)
    

In [39]:
company_dict = {k: v for k, v in company_dict.items() if v.condition}

In [40]:
len(company_dict)

111

In [41]:
sheet_ban_list = ['4대보험 체납', '종합평가', '임금체불']

In [42]:
for sheet_name in sheet_name_list:
    if sheet_name in sheet_ban_list:
        print(sheet_name)
        continue
    else:
        df_sheet = df_raw_data_dict[sheet_name]
        
        if pd.isna(pd.to_datetime(df_sheet.columns[-1], errors='coerce')):
            df_sheet = df_sheet.iloc[:, :-1]
        new_cols = (list(df_sheet.columns[:9]) + list(pd.to_datetime(df_temp.columns[9:]).map(lambda dt: dt.replace(day=1))))
        df_sheet.columns = new_cols
        df_raw_data_dict[sheet_name] = df_sheet
        
        for i, row in df_sheet.iterrows():
            if i + 1 in company_dict.keys():
                company_dict[i + 1].data_dict[sheet_name] = row[company_dict[i + 1].date_range].fillna(0).astype(float)
        

임금체불
4대보험 체납
종합평가
